In [11]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_exa import ExaSearchResults
from dotenv import load_dotenv
import os

from langchain_groq import ChatGroq


load_dotenv()

search_tool = ExaSearchResults(
    exa_api_key=os.environ["EXA_API_KEY"]
)

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

@tool("search", description="Search the web for information.")
def search(query):
    return search_tool._run(
        query=query,
        num_results=2,
        highlights=True,
    )
    


agent = create_agent(
    model=model,
    tools=[search],
    system_prompt=(
        "You are a helpful assistant. "
        "Use the search tool when you need current or factual information. "
        "Answer the user based on the search results."
    ),
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Who won the las world cup?",
        }
    ]
})
for msg in result["messages"]:
     print(msg.pretty_print())
print("***********************************")
print(result["messages"][-1].content)


================================ Human Message =================================

Who won the las world cup?
None
================================== Ai Message ==================================
Tool Calls:
  search (fc_ba339331-0f49-4e60-8e6f-4ad66c2505a1)
 Call ID: fc_ba339331-0f49-4e60-8e6f-4ad66c2505a1
  Args:
    query: 2022 FIFA World Cup winner
None
================================= Tool Message =================================
Name: search

Title: Messi wins World Cup, Argentina beats France on penalties | AP News
URL: https://apnews.com/article/lionel-messi-argentina-win-world-cup-final-against-france-e13fc1886725a0fe4f9e053e16a061bc
ID: https://apnews.com/article/lionel-messi-argentina-win-world-cup-final-against-france-e13fc1886725a0fe4f9e053e16a061bc
Score: None
Published Date: 2022-12-18T21:02:48.000Z
Author: Steve Douglas
Image: https://dims.apnews.com/dims4/default/f851554/2147483647/strip/true/crop/3000x2001+0+0/resize/1200x800!/quality/90/?url=https%3A%2F%2Fstorage.go

In [ ]:
### add short-term memory

from langgraph.checkpoint.memory import MemorySaver


memory = MemorySaver()
agent = create_agent(
    model=model,
    tools=[search],
    system_prompt=(
        "You are a helpful assistant. "
        "Use the search tool when you need current or factual information. "
        "Answer the user based on the search results."
    ),
    checkpointer=memory,
)

result = agent.invoke(
    {
    "messages": [
            {
                "role": "user",
                "content": "Who won the las world cup?",
            }
        ]
    },
    {"configurable": {"thread_id": "1b"}},
)

print(result["messages"][-1].content)

In [ ]:
### add long term memory
from langgraph.checkpoint.postgres import PostgresSaver  
import os
from dotenv import load_dotenv
from pprint import pprint

load_dotenv()

def get_user_info() -> str:
    """Look up information about the current user."""
    return "No user profile on file."

DB_URI = os.getenv("DB_URL")

with PostgresSaver.from_conn_string(DB_URI) as checkpointer:
    checkpointer.setup() # auto create tables in PostgreSQL
    agent = create_agent(
        model=model,
        tools=[get_user_info],
        checkpointer=checkpointer,
    )
    
    config = {
        "configurable": {"thread_id": "test"}
    }

    state = agent.get_state(config)  ## get agent state
    pprint(state)
    for message in state.values["messages"]:
        print(message.pretty_print())

StateSnapshot(values={'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='ab3c22e9-4a47-4f6a-84c7-0c6f0b44bb1d'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 385, 'total_tokens': 479, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'meta/muse-glimmer-30b', 'system_fingerprint': None, 'id': 'b827e0ef35ce3e02', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13acd-55f2-7080-9264-b6270595dd1c-0', tool_calls=[{'name': 'get_user_info', 'args': {}, 'id': 'chatcmpl-tool-93cadd39891fad05', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 385, 'output_tokens': 94, 'total_tokens': 479, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})

Blog Posts Maker

In [9]:
from langchain_core.prompts import PromptTemplate

human_style = """
Write in a natural, human way.

Sound like a knowledgeable person explaining something to another
person, not like an AI trying to sound professional.

Prefer:
- Clear and conversational language
- Natural sentence variation
- Direct explanations
- Specific examples when useful
- Natural transitions
- A confident but relaxed tone

Avoid:
- Generic introductions
- Empty statements and filler
- Excessive headings
- Repetitive conclusions
- Overly formal language
- Unnecessary buzzwords
- Phrases that sound like AI-generated marketing copy

Do not sacrifice clarity just to sound casual.
"""
outline_prompt = PromptTemplate.from_template("""
Create an outline for a blog post about:

Topic:
{topic}

Research:
{research}

The outline should have a logical progression and give the writer
enough direction to produce a useful article.

Include:
- A suitable title
- Introduction
- Main sections and the important points for each
- Conclusion

The article should be useful and specific rather than generic.

{human_style}

Do not write the actual article. Only produce the outline.
""")
draft_prompt = PromptTemplate.from_template("""
Write a complete blog post about:

Topic:
{topic}

Use this research:
{research}

Use this outline as the structure:
{outline}

Writing style:
{human_style}

Follow the outline, but don't make the article feel like you're
just filling in a template.

Explain ideas naturally and connect sections together.

Use practical examples where they make sense.

Do not invent facts that aren't supported by the research.

Write the complete draft.
""")
polish_prompt = PromptTemplate.from_template("""
Polish the following blog post.

Topic:
{topic}

Draft:
{draft}

Writing style:
{human_style}

Improve:
- Clarity
- Flow
- Grammar
- Sentence structure
- Transitions
- Repetition
- Readability

Keep the author's original ideas and meaning.

Don't rewrite it into a completely different article.
Don't add unsupported information.
Don't make it sound overly formal or artificially polished.

The final result should feel like something a knowledgeable person
would actually publish.

Return only the finished blog post.
""")

In [12]:
from langchain_openai import ChatOpenAI
from langchain_exa import ExaSearchResults
from langchain_core.tools import tool
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableLambda,
)
from langchain.agents import create_agent


#  Search tool
search_tool = ExaSearchResults(
    exa_api_key=os.environ["EXA_API_KEY"]
)

@tool("search", description="Search the web for information.")
def search(query: str) -> str:
    """Search the web and return relevant results."""
    return search_tool._run(
        query=query,
        num_results=2,
        highlights=True,
    )


#  Research agent
research_agent = create_agent(
    model=model,
    tools=[search],
    system_prompt="""
You are a research assistant for a blog-writing workflow.

Your job is to prepare useful research notes about the user's topic.

- If the topic is familiar, stable, and you can answer confidently,
  use your existing knowledge.
- If you are uncertain, need specific facts, or the topic involves
  recent developments, use the search tool.
- If you search, use the results to produce a concise summary of
  relevant findings.
- Do not invent facts, search results, or sources.
- Include useful source URLs when they are available.
- Return research notes that will help another model write a blog post.
- Do not write the full blog post yourself.
"""
)


#  Research runnable
def get_research(inputs):
    result = research_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": f"Research this blog topic: {inputs['topic']}"
            }
        ]
    })

    return result["messages"][-1].content


research_chain = RunnableLambda(get_research)


#  Writing chains
parser = StrOutputParser()

outline_chain = outline_prompt | model | parser

draft_chain = draft_prompt | model | parser

polish_chain = polish_prompt | model | parser


# Connect everything
blog_chain = (
    RunnablePassthrough.assign(
        research=research_chain
    )
    | RunnablePassthrough.assign(
        outline=outline_chain
    )
    | RunnablePassthrough.assign(
        draft=draft_chain
    )
    | RunnablePassthrough.assign(
        final_post=polish_chain
    )
    | RunnableLambda(lambda result: result["final_post"])
)


# Give the workflow a topic
topic = "LLMs"

final_post = blog_chain.invoke({
    "topic": topic,
    "human_style": human_style,
})

print(final_post)

**LLMs in 2024: The “Omni‑Capability” Year and What It Means for Builders**  

---

### 1. Why 2024 Feels Like a Turning Point  

I asked a model to read a 300‑page contract, pull out every liability clause, and rewrite the result in plain English. Within seconds it gave me a concise, accurate briefing—something that would have taken a junior associate an hour or more.  

That wasn’t a fluke. A series of upgrades over the past year turned large language models from clever chatbots into truly *omni‑capable* assistants. In plain terms, an LLM is a massive neural network (often billions of parameters) that has been trained on a huge mix of text, images, audio, and even video. By learning to predict the next token, the model picks up the ability to write prose, reason through problems, control external tools, and now perceive the world through multiple senses.  

Below I’ll walk through the biggest shifts that happened in 2024, point out the models that introduced them, and give you a prac

# SQL Task Manager

from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.agents import create_agent

db = SQLDatabase.from_uri("sqlite:///my_taks.db")

toolkit = SQLDatabaseToolkit(
    db=db,
    llm=model,
)

agent = create_agent(
    model=model,
    tools=toolkit.get_tools(),
    system_prompt="You are an assistant that works with my SQL database.",
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What tables are in my database?"
        }
    ]
})

print(result["messages"][-1].content)